In [8]:
from langchain.tools import tool


@tool
def backup_data(target: str) -> str:
    """备份数据， 敏感操作"""
    return f"Backup completed for {target}"


@tool
def clean_temp(folder: str) -> str:
    """清理临时目录"""
    return f"Clean completed for {folder}"


from langgraph.checkpoint.memory import MemorySaver

checkpoint = MemorySaver()

import os

import dotenv
from langchain.chat_models import init_chat_model

dotenv.load_dotenv()

DEEPSEEK_API = os.getenv("DEEPSEEK_API")

model = init_chat_model(
    model="openai:deepseek-v4-pro", api_key=DEEPSEEK_API, base_url="https://api.deepseek.com/v1", temperature=0.7
)

from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    tools=[backup_data, clean_temp],
    interrupt_on={
        "backup_data": True,
        "clean_temp": True,
    },
    system_prompt="""
    你是一名系统助手， 根据用户请求决定是否调用备份文件或清理工具。
    对于敏感操作会暂停并等待人工确认。
    """,
    checkpointer=checkpoint,
)

config = {"configurable": {"thread_id": "test6"}}

from langchain.messages import HumanMessage
from langgraph.types import Command

result = agent.invoke(
    {"messages": [HumanMessage(content="备份 data.db 文件到当前目录， 文件名为data.db.bak; 并清理下 /temp 目录")]},
    config=config,
)

print(result.keys())
print(result.get("__interrupt__"))
while interrupt := result.get("__interrupt__"):
    interrupt_value = result.get("__interrupt__")[0].value
    action_requests = interrupt_value["action_requests"]

    decisions = []
    print("1111")
    for ar in action_requests:
        print(f"\n工具: {ar['name']}, 参数: {ar['args']}")
        decision = "approve" if input("input: y/n") == "y" else "reject"
        decisions.append({"type": decision})
    print("1111")
    result = agent.invoke(Command(resume={"decisions": decisions}), config=config)
for msg in result["messages"]:
    print(msg)
# while result.get("__interrupt__"):
#     print(result["__interrupt__"])
#     break

dict_keys(['messages', 'files', '__interrupt__'])
[Interrupt(value={'action_requests': [{'name': 'backup_data', 'args': {'target': 'data.db'}, 'description': "Tool execution requires approval\n\nTool: backup_data\nArgs: {'target': 'data.db'}"}, {'name': 'clean_temp', 'args': {'folder': '/temp'}, 'description': "Tool execution requires approval\n\nTool: clean_temp\nArgs: {'folder': '/temp'}"}], 'review_configs': [{'action_name': 'backup_data', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}, {'action_name': 'clean_temp', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='aa5d835dc75200f27633991c60118433')]
1111

工具: backup_data, 参数: {'target': 'data.db'}

工具: clean_temp, 参数: {'folder': '/temp'}
1111
content='备份 data.db 文件到当前目录， 文件名为data.db.bak; 并清理下 /temp 目录' additional_kwargs={} response_metadata={} id='78c70ded-ccff-4c78-85c1-0e4893d54c3b'
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 168, 'prom